# 07 - Evaluation & insights

In [1]:
import glob, json
import pandas as pd

rows = [json.load(open(f)) for f in sorted(glob.glob("../results/*.json"))]
df = pd.DataFrame(rows).set_index("name")

def col(c):
    """Colonna se esiste (solo alcune righe l'hanno), altrimenti tutta NaN."""
    return df[c] if c in df else pd.Series(float("nan"), index=df.index)

ref = df.loc["teacher_baseline"]

is_sparse = col("nonzero_params").notna()                   # righe del pruning non strutturato
params = col("nonzero_params").fillna(df["params"]).astype(int)   # non nulli per le righe sparse
macs_dense = col("macs_dense").fillna(df["macs"])           # df["macs"] è già sparso per le righe sparse

table = pd.DataFrame({
    "test acc (%)": (df["test_acc"] * 100).round(2),
    "Δ acc (pt)": ((df["test_acc"] - ref["test_acc"]) * 100).round(2),
    "params (non nulli)": params,
    "sparsity (%)": (col("sparsity") * 100).round(1),
    "size (MB)": df["size_mb"].round(3),
    "size x": (ref["size_mb"] / df["size_mb"]).round(1),
    "MACs (M)": (df["macs"] / 1e6).round(2),
    "MACs denso (M)": (macs_dense / 1e6).round(2),
    "Peak RAM (KB)": df["peak_act_kb"].round(2),
    "latency (ms)": df["latency_ms_median"].round(2),   # CPU, 1 thread
    "energy (µJ)": df["energy_uj"].round(2),
})
table.index = [n + " *" if s else n for n, s in zip(table.index, is_sparse)]

print("* pruning non strutturato: params = pesi non nulli; MACs ed energia sono sparsi \n"
      "(validi solo su HW che salta gli zeri); size = stima formato sparso; \n"
      "latency non misurata (con tensori densi non cambia).\n")
table.style.format(precision=2, na_rep="–")

* pruning non strutturato: params = pesi non nulli; MACs ed energia sono sparsi 
(validi solo su HW che salta gli zeri); size = stima formato sparso; 
latency non misurata (con tensori densi non cambia).



,test acc (%),Δ acc (pt),params (non nulli),sparsity (%),size (MB),size x,MACs (M),MACs denso (M),Peak RAM (KB),latency (ms),energy (µJ)
student_baseline,80.55,-8.73,245226,–,1.00,61.90,3.56,3.56,45.06,0.69,16.40
student_int8,80.39,-8.89,244538,–,0.25,248.90,3.56,3.56,11.26,–,1.07
student_kd,80.04,-9.24,245226,–,1.00,61.90,3.56,3.56,45.06,0.70,16.40
student_kd_struct,79.66,-9.62,186362,–,0.75,82.50,3.51,3.51,45.06,0.83,16.13
student_kd_struct_int8,79.67,-9.61,186362,–,0.19,326.80,3.51,3.51,11.26,–,1.05
student_lowrank,78.60,-10.68,231994,–,0.94,66.30,3.38,3.38,45.06,0.67,15.53
student_struct,78.77,-10.51,186362,–,0.75,82.50,3.51,3.51,45.06,0.66,16.13
student_unstruct *,78.52,-10.76,147018,40.00,0.88,70.60,2.54,3.56,45.06,–,11.67
teacher_baseline,89.28,0.00,15530762,–,62.16,1.00,215.69,215.69,327.68,8.00,992.17
teacher_combo_struct_int8,87.90,-1.38,10146885,–,10.17,6.10,210.31,210.31,81.92,–,63.09
